# Module 5.2: Unified Multi-Node Observability & Health Monitoring with Uptime Kuma

## Homelab Expansion (Chapter 2) - Step 5: Core Raspberry Pi Infrastructure Services

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand why an observability and alerting service (**Uptime Kuma**) should run on the low-power core node (`homelab-pi`) to monitor all worker nodes and microservices across the `USW-Enterprise-8-PoE` switch.
2. Deploy **Uptime Kuma** on `homelab-pi` (`10.0.0.200:3001`) with persistent storage on `/mnt/homelab_storage`.
3. Configure real-time HTTP, TCP, DNS, and ICMP Ping monitors for both `homelab-pi` and `homelab-gpu` (including the remote CUDA Machine Learning `/ping` endpoint on `10.0.0.201:3003`).
4. Review the complete two-node homelab operational checklist.

## 5.2.1 Conceptual Background: Watching the Watchmen in a Multi-Node Cluster

Now that your homelab spans two physical machines (`homelab-pi` at `10.0.0.200` and `homelab-gpu` at `10.0.0.201`) connected over your `USW-Enterprise-8-PoE` multi-gig switch, you need a single pane of glass to verify that every host, port, and container is healthy.

```mermaid
flowchart LR
    Kuma["Uptime Kuma on homelab-pi<br>http://10.0.0.200:3001"]
    
    Kuma -->|"HTTP GET :2283/api/server/ping"| Immich["Immich Core API (10.0.0.200:2283)"]
    Kuma -->|"DNS Query :53 (github.com)"| Pihole["Pi-hole v6 DNS (10.0.0.200:53)"]
    Kuma -->|"ICMP Ping"| Laptop["Node 2: homelab-gpu (10.0.0.201)"]
    Kuma -->|"HTTP GET :3003/ping (Expect 'pong')"| CUDAML["Immich CUDA ML Worker (10.0.0.201:3003)"]
```

By hosting **Uptime Kuma** on `homelab-pi`:
- It consumes only **~65 MB of RAM** and `<1%` CPU.
- If `homelab-gpu` ever goes offline, restarts, or stops responding on port `3003`, `homelab-pi` stays online to record the outage timestamp, track historical latency graphs across the switch, and optionally push an instant notification to your phone via Discord, Telegram, Ntfy, or Email.

## 5.2.2 Step-by-Step Laboratory Guide

### Task 5.2.1: Deploy Uptime Kuma on `homelab-pi`
Connect to your Raspberry Pi (`ssh homelab-pi`) and create the Uptime Kuma stack directory mapped to your external NVMe SSD:
```bash
mkdir -p ~/homelab/stacks/uptime-kuma
mkdir -p /mnt/homelab_storage/docker_volumes/uptime-kuma
cd ~/homelab/stacks/uptime-kuma
```

Create the `docker-compose.yml` file:
```bash
cat << 'EOT' > ~/homelab/stacks/uptime-kuma/docker-compose.yml
name: uptime_kuma

services:
  uptime-kuma:
    container_name: uptime_kuma
    image: louislam/uptime-kuma:1
    ports:
      - "3001:3001"
    volumes:
      - /mnt/homelab_storage/docker_volumes/uptime-kuma:/app/data
      # Optional: Mount docker.sock read-only to monitor local Pi containers directly!
      - /var/run/docker.sock:/var/run/docker.sock:ro
    environment:
      - TZ=America/New_York
    restart: unless-stopped
EOT
```

Launch the container and verify it is running:
```bash
docker compose up -d
docker compose ps
```

### Task 5.2.2: Configure Multi-Node Health Monitors in Uptime Kuma
1. Open your browser and navigate to:
   ```text
   http://10.0.0.200:3001
   ```
   (or `http://homelab-pi:3001` over Tailscale).
2. Create your administrator account on the initial setup screen.
3. Click **+ Add New Monitor** to register the four pillars of your hybrid homelab:

| Monitor Friendly Name | Monitor Type | Target URL / Hostname | Port / Expected Keyword | Heartbeat Interval |
| :--- | :--- | :--- | :--- | :---: |
| **1. Node 2 (`homelab-gpu`) Host Ping** | `Ping` | `10.0.0.201` | Sub-ms LAN ICMP ping | `60s` |
| **2. Immich CUDA ML Worker (`homelab-gpu`)** | `HTTP(s) - Keyword` | `http://10.0.0.201:3003/ping` | Keyword: `pong` | `60s` |
| **3. Immich Web & Core API (`homelab-pi`)** | `HTTP(s) - Keyword` | `http://10.0.0.200:2283/api/server/ping` | Keyword: `pong` | `60s` |
| **4. Pi-hole v6 DNS Resolver (`homelab-pi`)** | `DNS` | `10.0.0.200` (Resolve: `github.com`, Record: `A`) | Port `53` | `60s` |

### 🖼️ Task 5.2.2 Verification: Unified Multi-Node Uptime Kuma Dashboard

**Verification Objective:** Capture a screenshot of your Uptime Kuma dashboard (`http://10.0.0.200:3001`) showing `100%` green status bars across both `homelab-pi` (`10.0.0.200`) and `homelab-gpu` (`10.0.0.201`).

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Chapter 2 Final Checkpoint & Reflection

Congratulations on expanding your single-board Raspberry Pi NAS into a multi-node, GPU-accelerated hybrid homelab! Confirm your final Chapter 2 milestones below:

- [ ] **Hardware & Battery Bypass:** 2019 ROG Zephyrus degraded Li-Po battery safely removed, motherboard header insulated with Kapton tape, thermal paste (`MX-6`/`NT-H2`/`PTM7950`) & VRM thermal putty (`K5 PRO`) refreshed, and protected by an external UPS.
- [ ] **Headless Server OS:** Unlicensed Windows 11 wiped and replaced with bare-metal **Ubuntu Server 24.04 LTS (`x86_64`)**, configured with `HandleLidSwitch=ignore` and `consoleblank=60`.
- [ ] **Unified `USW-Enterprise-8-PoE` Switch Topology:** Both `homelab-pi` (`10.0.0.200`) and `homelab-gpu` (`10.0.0.201`) connected to the Ubiquiti `USW-Enterprise-8-PoE` (8× 2.5GbE PoE+ + 2× 10G SFP+) achieving full wire-speed node-to-node throughput (`~940 Mbps` to the Pi) and joined to your Tailscale mesh.
- [ ] **GPU Workload Migration:** NVIDIA headless driver, `nvidia-homelab-powercap.service`, and `nvidia-container-toolkit` running `immich-machine-learning:release-cuda` on `homelab-gpu:3003`, freeing ~1.2GB+ of RAM on `homelab-pi`.
- [ ] **Core Raspberry Pi Infrastructure:** **Pi-hole v6** (`:53` / `:8080`) providing LAN and global Tailscale ad-blocking, and **Uptime Kuma** (`:3001`) monitoring every node and service in your cluster.

Ready for Enterprise Storage & UniFi Ecosystem Management? Proceed to [Chapter 3: UniFi UNAS 2 Storage & Enterprise Networking](../../03_unifi_nas_and_enterprise_networking/README.md)!